# 03 - Apply the unlearning methods

Stage 5: six methods x eight conditions x five seeds = 240 runs. Requires stage 3 to have finished -- an unlearning run needs the original model it modifies, and will fail loudly rather than unlearn from a fresh initialisation.

**Set `ACCOUNT` below** to this account's number. Every account runs the identical notebook with
a different `ACCOUNT`, and they compute disjoint halves of the work with no coordination -- run
identifiers are pure functions of the config, so all accounts derive the same work list and each
takes its own stripe of it.

Runs already present in the attached artefact dataset are skipped, so a session that dies costs
only its in-flight model.


In [ ]:
# --- clone the repo at a PINNED commit ------------------------------------------------------
# Clone rather than `pip install git+...`. A wheel would contain only src/forgetcheck/, but the
# CLI also needs configs/ (the metric registry, seed streams, audit protocols) and
# data/memorization/ (the RUM scores -- 400 KB, committed precisely so a fresh session does not
# have to re-download 2 GB from Google Drive).
#
# Pinning is what makes provenance work: every record this session writes carries this commit,
# so any result can be traced back to the exact code that produced it.
REPO   = "https://github.com/hyperreal2005/Minor-Project.git"
COMMIT = "main"          # <-- pin to a sha for real runs, e.g. "a1b2c3d"

import os
from pathlib import Path

os.chdir("/kaggle/working")
if not Path("Minor-Project").exists():
    !git clone --quiet $REPO
%cd /kaggle/working/Minor-Project
!git fetch --quiet --all && git checkout --quiet $COMMIT
!git log -1 --format="pinned at %h  %s"
!pip install -q -e .


In [ ]:
from pathlib import Path

REPO_DIR = Path("/kaggle/working/Minor-Project")

# Attach a previous artefact dataset (Add Input -> Datasets) so completed runs are skipped
# rather than recomputed. Without it, every session starts from nothing.
ARTIFACTS_IN = Path("/kaggle/input/forgetcheck-artifacts")
if ARTIFACTS_IN.exists():
    !cp -r $ARTIFACTS_IN/artifacts $REPO_DIR/ 2>/dev/null || true
    !cp -r $ARTIFACTS_IN/results   $REPO_DIR/ 2>/dev/null || true
    print("restored artefacts from a previous session")
else:
    print("no previous artefacts attached - starting fresh")

import torch
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))
else:
    print("!! running on CPU. Set Settings -> Accelerator -> GPU.")
    print("!! On CPU one 30-epoch training run takes ~4.8 hours instead of ~12 minutes.")


In [ ]:
ACCOUNT = 1     # <-- this account's number, 1-based
OF      = 3     # <-- how many accounts are sharing this stage

DEVICE = "cuda" if __import__("torch").cuda.is_available() else "cpu"
STAGE  = 5
print(f"account {ACCOUNT} of {OF}, stage {STAGE}, device {DEVICE}")


## What this account will do

In [ ]:
!forgetcheck --root . --dry-run queue --stage {STAGE} --account {ACCOUNT} --of {OF}


## Run it

This is the long cell. It prints each run as it starts and finishes, so you can watch progress and estimate the remaining time.

In [ ]:
!forgetcheck --root . --device {DEVICE} queue --stage {STAGE} --account {ACCOUNT} --of {OF}


## Inspect what came out

Watch `neggrad` specifically: it is the **destructive control** and is expected to wreck retain accuracy while driving forget accuracy down. That is the point of including it, not a bug.

In [ ]:
from forgetcheck.registry import read_records
import pandas as pd

df = read_records("results/records")
print(f"{len(df)} rows across {df['run_id'].nunique()} runs\n")

wide = df.pivot_table(index=["run_id", "role"], columns="metric", values="value")
wide.round(4)


In [ ]:
# Seed-to-seed spread. This is not a diagnostic to average away -- it is the raw material for
# every oracle band in the calibration stage, and stage 3's gate is that it stays within 0.5 pp.
acc = df[df.metric == "test_acc"]
if len(acc) > 1:
    spread = acc.groupby("forget_id")["value"].agg(["mean", "std", "count"])
    print(spread.round(4))


In [ ]:
!forgetcheck --root . status


In [ ]:
# --- push the artefacts back out -------------------------------------------------------------
# Kaggle sessions are disposable. Anything not saved to a Dataset version is gone, and the next
# session would recompute it. /kaggle/working persists per notebook (~20 GB); a Dataset is what
# shares it between notebooks and accounts.
import json
from pathlib import Path

OUT = Path("/kaggle/working/to_upload")
OUT.mkdir(exist_ok=True)
!cp -r /kaggle/working/Minor-Project/artifacts $OUT/ 2>/dev/null || true
!cp -r /kaggle/working/Minor-Project/results   $OUT/ 2>/dev/null || true

META = OUT / "dataset-metadata.json"
META.write_text(json.dumps({
    "title": "forgetcheck-artifacts",
    "id": "YOUR-KAGGLE-USERNAME/forgetcheck-artifacts",   # <-- your username
    "licenses": [{"name": "CC0-1.0"}],
}, indent=2))

# Needs an API token at ~/.kaggle/kaggle.json (Kaggle -> Account -> Create New API Token).
# First time:   !kaggle datasets create  -p $OUT --dir-mode zip
# Afterwards:   !kaggle datasets version -p $OUT -m "stage N account K" --dir-mode zip
#
# Simplest alternative: just "Save Version" the notebook. /kaggle/working persists per notebook
# (~20 GB), which is enough for one account to resume itself -- but a Dataset is what shares
# artefacts between accounts, and that is what the team needs.
print(f"staged in {OUT} - uncomment whichever line above applies")
